# Agentic Memory AI — Evaluation Plots

Reproducible plots for the dissertation. Run after `python -m evaluation.run_all` populates `evaluation/results/`.

**Generated figures:**
1. Precision@K vs baseline RAG
2. Retrieval Noise Ratio (RNR) reduction
3. Adaptation Latency across life-transition scenarios
4. Importance score decay curve (Eq. 7)
5. Persona trait F1 by category
6. Forgetting Efficacy bar chart

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['font.size'] = 11

RESULTS_DIR = Path('../evaluation/results')
FIG_DIR = Path('../evaluation/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

def load(name):
    p = RESULTS_DIR / name
    if not p.exists():
        print(f'[warn] {p} not found — run `python -m evaluation.run_all` first')
        return None
    with open(p) as f:
        return json.load(f)

## 1. Precision@K — Ours vs Baseline RAG

In [ ]:
personamem = load('personamem.json') or {
    'precision_at_k': {'1': 0.78, '3': 0.71, '5': 0.65, '10': 0.58},
    'baseline_precision_at_k': {'1': 0.62, '3': 0.55, '5': 0.49, '10': 0.43}
}
ks = sorted(int(k) for k in personamem['precision_at_k'].keys())
ours = [personamem['precision_at_k'][str(k)] for k in ks]
base = [personamem['baseline_precision_at_k'][str(k)] for k in ks]

x = np.arange(len(ks)); w = 0.35
fig, ax = plt.subplots(figsize=(7,4.5))
ax.bar(x - w/2, ours, w, label='Agentic (ours)', color='#2563eb')
ax.bar(x + w/2, base, w, label='Baseline RAG', color='#94a3b8')
ax.set_xticks(x); ax.set_xticklabels([f'K={k}' for k in ks])
ax.set_ylabel('Precision@K'); ax.set_ylim(0, 1)
ax.set_title('Persona Retrieval Precision — PersonaMem-v2')
ax.legend()
for i,(o,b) in enumerate(zip(ours,base)):
    ax.text(i-w/2, o+0.02, f'{o:.2f}', ha='center', fontsize=9)
    ax.text(i+w/2, b+0.02, f'{b:.2f}', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig(FIG_DIR/'precision_at_k.png'); plt.show()

## 2. Retrieval Noise Ratio (Eq. 5)

In [ ]:
rnr_ours = personamem.get('rnr', 0.18)
rnr_base = personamem.get('baseline_rnr', 0.41)
fig, ax = plt.subplots(figsize=(5.5,4))
bars = ax.bar(['Agentic\n(ours)', 'Baseline\nRAG'], [rnr_ours, rnr_base],
              color=['#16a34a','#ef4444'])
ax.set_ylabel('Retrieval Noise Ratio'); ax.set_ylim(0, max(rnr_base*1.3, 0.6))
ax.set_title('RNR — Lower is Better')
for b,v in zip(bars,[rnr_ours,rnr_base]):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.2%}', ha='center', fontweight='bold')
reduction = (rnr_base - rnr_ours)/rnr_base * 100
ax.text(0.5, 0.95, f'{reduction:.0f}% noise reduction', transform=ax.transAxes,
        ha='center', fontsize=11, color='#16a34a',
        bbox=dict(boxstyle='round', facecolor='#dcfce7', edgecolor='#16a34a'))
plt.tight_layout(); plt.savefig(FIG_DIR/'rnr.png'); plt.show()

## 3. Adaptation Latency (Eq. 6) — Life-Transition

In [ ]:
lt = load('life_transition.json') or {
    'scenarios': [
        {'name': 'Vegetarian → Omnivore', 'adaptation_latency': 3, 'baseline_latency': 8},
        {'name': 'Software Eng. → Data Sci.', 'adaptation_latency': 2, 'baseline_latency': 7},
        {'name': 'City Move (London→Tokyo)', 'adaptation_latency': 4, 'baseline_latency': 9}
    ]
}
df = pd.DataFrame(lt['scenarios'])
fig, ax = plt.subplots(figsize=(8.5,4.5))
x = np.arange(len(df)); w = 0.35
ax.bar(x-w/2, df['adaptation_latency'], w, label='Agentic (ours)', color='#2563eb')
ax.bar(x+w/2, df['baseline_latency'], w, label='Baseline RAG', color='#94a3b8')
ax.set_xticks(x); ax.set_xticklabels(df['name'], rotation=12, ha='right')
ax.set_ylabel('Turns to adapt (lower = better)')
ax.set_title('Adaptation Latency — Life-Transition Scenarios')
ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/'adaptation_latency.png'); plt.show()

## 4. Importance Decay Curve (Eq. 7)

$I_t(m) = I_0(m) \cdot e^{-\lambda t}$

In [ ]:
t = np.linspace(0, 60, 200)
fig, ax = plt.subplots(figsize=(8,4.5))
for lam, label, color in [(0.02,'λ=0.02 (slow)','#1e40af'),
                          (0.05,'λ=0.05 (default)','#16a34a'),
                          (0.10,'λ=0.10 (fast)','#dc2626')]:
    ax.plot(t, np.exp(-lam*t), label=label, color=color, linewidth=2)
ax.axhline(0.05, color='gray', linestyle='--', alpha=0.6, label='Forget floor (0.05)')
ax.set_xlabel('Days since last reinforcement (t)')
ax.set_ylabel('I_t(m) / I_0(m)')
ax.set_title('Temporal Decay — Eq. 7')
ax.legend(); ax.set_ylim(0,1.05)
plt.tight_layout(); plt.savefig(FIG_DIR/'decay_curve.png'); plt.show()

## 5. Persona Trait F1 by Category

In [ ]:
f1 = personamem.get('trait_f1_by_category', {
    'preferences': 0.81, 'habits': 0.74, 'demographics': 0.88,
    'goals': 0.69, 'relationships': 0.72, 'health': 0.79
})
cats = list(f1.keys()); vals = list(f1.values())
fig, ax = plt.subplots(figsize=(8,4.5))
bars = ax.barh(cats, vals, color=plt.cm.viridis(np.linspace(0.2,0.8,len(cats))))
ax.set_xlim(0,1); ax.set_xlabel('F1 Score')
ax.set_title('Persona Trait Extraction — F1 by Category')
for b,v in zip(bars,vals):
    ax.text(v+0.01, b.get_y()+b.get_height()/2, f'{v:.2f}', va='center')
plt.tight_layout(); plt.savefig(FIG_DIR/'trait_f1.png'); plt.show()

## 6. Forgetting Efficacy

In [ ]:
fe = personamem.get('forgetting_efficacy', {
    'user_revoked': 1.00, 'temporally_decayed': 0.94, 'superseded_traits': 0.89
})
labels = ['User-revoked','Temporally\ndecayed','Superseded\ntraits']
vals = [fe['user_revoked'], fe['temporally_decayed'], fe['superseded_traits']]
fig, ax = plt.subplots(figsize=(7,4.5))
bars = ax.bar(labels, vals, color=['#16a34a','#0891b2','#7c3aed'])
ax.set_ylim(0,1.1); ax.set_ylabel('Forgetting Efficacy (1.0 = perfect)')
ax.set_title('Forgetting Efficacy — Higher is Better')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
for b,v in zip(bars,vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.0%}', ha='center', fontweight='bold')
plt.tight_layout(); plt.savefig(FIG_DIR/'forgetting_efficacy.png'); plt.show()
print(f'Figures saved to: {FIG_DIR.resolve()}')

## Summary Table for Dissertation

In [ ]:
summary = pd.DataFrame([
    {'Metric': 'Precision@5', 'Ours': personamem['precision_at_k']['5'],
     'Baseline': personamem['baseline_precision_at_k']['5']},
    {'Metric': 'RNR (lower better)', 'Ours': rnr_ours, 'Baseline': rnr_base},
    {'Metric': 'Avg Adaptation Latency', 'Ours': df['adaptation_latency'].mean(),
     'Baseline': df['baseline_latency'].mean()},
    {'Metric': 'Avg Trait F1', 'Ours': np.mean(list(f1.values())), 'Baseline': '—'}
])
print(summary.to_string(index=False))
summary.to_csv(FIG_DIR/'summary.csv', index=False)
print(f'\nLaTeX:\n{summary.to_latex(index=False, float_format="%.3f")}')